In [ ]:
%%capture
!pip install yfinance pandas scikit-learn tensorflow requests

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import pickle
from datetime import datetime, timedelta
import time
import requests
import io
import os

print("Libraries loaded!")

In [ ]:
BACKEND_URL = "http://localhost:8000"  
TRAINING_INTERVAL_HOURS = 24  
NUM_MODELS = 3  
DAYS_OF_DATA = 365  

print(f"Backend URL: {BACKEND_URL}")
print(f"Training interval: {TRAINING_INTERVAL_HOURS} hours")
print(f"Number of models: {NUM_MODELS}")

In [ ]:
def create_sequences(data, window_size=60):
    xs, ys = [], []
    for i in range(len(data) - window_size):
        xs.append(data[i:(i + window_size)])
        ys.append(data[i + window_size][0])  # Predict Close price
    return np.array(xs), np.array(ys)

def build_model(input_shape, units=64):
    model = Sequential([
        LSTM(units, return_sequences=True, input_shape=input_shape),
        Dropout(0.2),
        LSTM(units//2, return_sequences=False),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])
    return model

def fetch_data(ticker, days):
    end_date = datetime.now()
    start_date = end_date - timedelta(days=days)
    print(f"  Fetching {ticker} data from {start_date.date()} to {end_date.date()}...")
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    if df.empty:
        raise RuntimeError("Failed to fetch data")
    print(f"  Downloaded {len(df)} rows")
    return df

In [ ]:
def train_single_model(df, model_id, units=64):
    print(f"\n[Model {model_id}] Training with {units} LSTM units...")
    features = df[['Close', 'Volume']].values.astype(np.float32)
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled = scaler.fit_transform(features)
    X, y = create_sequences(scaled, 60)
    print(f"  Sequences shape: {X.shape}")
    model = build_model((X.shape[1], X.shape[2]), units)
    early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)
    history = model.fit(X, y, epochs=30, batch_size=32, callbacks=[early_stop], verbose=0)
    print(f"  Final loss: {history.history['loss'][-1]:.6f}")
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.h5', delete=False) as tmp:
        model.save(tmp.name)
        with open(tmp.name, 'rb') as f:
            model_bytes = f.read()
        os.unlink(tmp.name)
    scaler_bytes = pickle.dumps(scaler)
    return model_bytes, scaler_bytes, float(history.history['loss'][-1])

print("Training function ready!")

In [ ]:
def send_model_to_backend(model_bytes, scaler_bytes, model_id, loss, backend_url):
    url = f"{backend_url}/upload-model"
    files = {
        'model': ('model.h5', model_bytes, 'application/octet-stream'),
        'scaler': ('scaler.pkl', scaler_bytes, 'application/octet-stream')
    }
    data = {
        'model_id': model_id,
        'loss': loss
    }
    try:
        resp = requests.post(url, files=files, data=data, timeout=30)
        if resp.status_code == 200:
            print(f"  [Model {model_id}] Sent to backend successfully!")
            return True
        else:
            print(f"  [Model {model_id}] Failed: {resp.status_code} - {resp.text}")
            return False
    except Exception as e:
        print(f"  [Model {model_id}] Error: {e}")
        return False

print("Upload function ready!")

In [ ]:
print("\n" + "="*60)
print("Bitcoin LSTM Training Loop Started!")
print(f"Backend: {BACKEND_URL}")
print(f"Retrain every: {TRAINING_INTERVAL_HOURS} hours")
print("="*60 + "\n")
while True:
    iteration += 1
    print(f"\n{'#'*20} ITERATION #{iteration} {'#'*20}")
    print(f"Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    try:
        print("\n[1/3] Fetching real-time data...")
        df = fetch_data("BTC-USD", DAYS_OF_DATA)
        print(f"\n[2/3] Training {NUM_MODELS} models...")
        for i in range(NUM_MODELS):
            units = [64, 96, 128][i] 
            model_id = f"model_{i+1}_lstm{units}"
            
            model_bytes, scaler_bytes, loss = train_single_model(df, model_id, units)
            
            print(f"  Sending {model_id} to backend...")
            send_model_to_backend(model_bytes, scaler_bytes, model_id, loss, BACKEND_URL)
            time.sleep(2)
        
    except Exception as e:
        print(f"\nERROR in iteration #{iteration}: {e}")
        import traceback
        traceback.print_exc()